# Data Cleaning - leesmateriaal.csv

This notebook cleans the reading catalogue (`leesmateriaal.csv`) step by step so it can be used for the recommender.

Steps:
1. Load and inspect the data
2. Drop empty / redundant / administrative columns
3. Rename columns to consistent snake_case
4. Normalize categorical values (capitals, spelling variants)
5. Fix data types and impossible values
6. Repair the URL columns for online articles and blogposts
7. Clean themes and keywords
8. Handle duplicates
9. Handle missing values
10. Final check and save

## 1) Load and inspect the data
The file is comma-separated and UTF-8 encoded.

In [1]:
# --- Load and inspect the data ---
import pandas as pd
import numpy as np
import re

df = pd.read_csv("../datasets/leesmateriaal.csv")

print("Shape:", df.shape)
df.head()

Shape: (283, 33)


,Titel,Auteur,Korte omschrijving,Unnamed: 3,Type materiaal,Niveau,Themas_boek,materiaal_id,genre,onderwerp,...,interessecluster,uitgever,publicatiejaar,beschikbaar_formaat,cataloguscode,interne_batchcode,administratieve_prioriteit,doelgroep_extra,genre_kopie,Gemiddelde beoordeling
0,Het dagboek van Anne Frank,Anne Frank,Dagboek van een Joods meisje dat tijdens de Tw...,NaN,boek,3F,WOII;onderduik,LM-0001,fictie,WOII,...,VERHALEN,Educatief Media,2025,PDF,CAT-00001,B1,Hoog,NaN,fictie,3.4
1,Blauw Water,S. van der Vlugt,Spannende thriller waarin een vrouw wordt geco...,NaN,Boek,3F,spanning;stalking,LM-0002,fictie,spanning,...,verhalen,Leeswereld,2005,Digitaal,CAT-00002,B2,hoog,NT2,fictie,2.5
2,Boer vindt vrouw,H. van der Werf,"Toegankelijk verhaal over liefde, relaties en ...",NaN,Boek,2F,liefde;relaties,LM-0003,fictie,liefde,...,verhalen,NaN,2025,Digitaal,CAT-00003,B2,Midden,NT2,fictie,3.8
3,Cadeau voor u!,Harry Mulisch,Bundel toegankelijke verhalen over verrassende...,NaN,Boek,3F,ontmoeting;verbondenheid,LM-0004,fictie,ontmoeting,...,verhalen,Uitgeverij Noord,2018,Digitaal,CAT-00004,batch-2,Laag,NT2,fictie,3.9
4,De biecht,Mickey Roorda,Verhaal waarin verborgen geheimen en persoonli...,NaN,Boek,2F,geheimen;schuld,LM-0005,fictie,geheimen,...,verhalen,Uitgeverij Noord,NaN,online,CAT-00005,B3,Laag,MBO/NT2,fictie,3.1


In [2]:
# Data types and number of missing values per column
overview = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "unique": df.nunique(),
})
overview

,dtype,missing,unique
Titel,str,0,277
Auteur,str,7,231
Korte omschrijving,str,0,282
Unnamed: 3,float64,283,0
Type materiaal,str,1,10
Niveau,str,6,4
Themas_boek,str,134,133
materiaal_id,str,0,279
genre,str,26,3
onderwerp,str,181,47


## 2) Drop empty / redundant / administrative columns
- `Unnamed: 3` is completely empty.
- `genre_kopie` is an exact copy of `genre` (and missing in the same rows), so it adds no information.
- `cataloguscode`, `interne_batchcode`, `administratieve_prioriteit` and `gegevensstatus` are administrative and say nothing about the content.

In [3]:
# --- Drop columns we don't need ---

# 1) Check that genre_kopie really is a copy of genre (where both are filled)
both = df["genre"].notna() & df["genre_kopie"].notna()
print("Rows where genre != genre_kopie:", (df.loc[both, "genre"] != df.loc[both, "genre_kopie"]).sum())
print("Rows where only genre_kopie is filled:", (df["genre"].isna() & df["genre_kopie"].notna()).sum())

# 2) Drop the columns
drop_cols = [
    "Unnamed: 3", "genre_kopie",
    "cataloguscode", "interne_batchcode", "administratieve_prioriteit", "gegevensstatus",
]
df = df.drop(columns=drop_cols)
print("Shape after dropping:", df.shape)

Rows where genre != genre_kopie: 0
Rows where only genre_kopie is filled: 0
Shape after dropping: (283, 27)


## 3) Rename columns
Some columns use capitals and spaces, others snake_case. We make them all snake_case.

In [4]:
# --- Consistent column names ---
df = df.rename(columns={
    "Titel": "titel",
    "Auteur": "auteur",
    "Korte omschrijving": "omschrijving",
    "Type materiaal": "type_materiaal",
    "Niveau": "niveau",
    "Themas_boek": "themas",
    "Gemiddelde beoordeling": "gem_beoordeling",
})
df.columns.tolist()

['titel',
 'auteur',
 'omschrijving',
 'type_materiaal',
 'niveau',
 'themas',
 'materiaal_id',
 'genre',
 'onderwerp',
 'trefwoorden',
 'taal',
 'aantal_paginas',
 'geschatte_leestijd_min',
 'bron_url',
 'doelgroep',
 'woordenaantal_omschrijving',
 'gem_woordlengte',
 'gem_zinslengte_woorden',
 'aandeel_lange_woorden_pct',
 'aantal_zinnen',
 'leescomplexiteit_score',
 'interessecluster',
 'uitgever',
 'publicatiejaar',
 'beschikbaar_formaat',
 'doelgroep_extra',
 'gem_beoordeling']

## 4) Normalize categorical values
Many columns contain the same value written in different ways (e.g. `Papier` / `papier`, `NL` / `NED` / `Nederlands`).
First we strip whitespace and convert to lowercase, then we map the remaining variants to one value.

In [5]:
# --- Normalize text categories ---
cat_cols = ["type_materiaal", "niveau", "genre", "taal", "doelgroep",
            "interessecluster", "uitgever", "beschikbaar_formaat"]

# Show the variants before cleaning
for col in cat_cols:
    print(col, "->", df[col].dropna().unique().tolist())

type_materiaal -> ['boek', 'Boek', 'tijdschrift', 'krant (papier)', 'digitale krant', 'Boek - thriller', 'Boek - roman', 'online artikel', 'dichtbundel', 'blogpost']
niveau -> ['3F', '2F', '3F+', '2F-3F']
genre -> ['fictie', 'informatief', 'poëzie']
taal -> ['Nederlands', 'NED', 'NL', 'nederlands']
doelgroep -> ['MBO', 'MBO/NT2', 'NT2', 'mbo', 'mbo / nt2']
interessecluster -> ['VERHALEN', 'verhalen', 'vrije tijd', 'maatschappij']
uitgever -> ['Educatief Media', 'Leeswereld', 'Uitgeverij Noord']
beschikbaar_formaat -> ['PDF', 'Digitaal', 'online', 'papier', 'Papier', 'Online']


In [6]:
# 1) Strip whitespace and convert to lowercase (titles/authors keep their capitals)
for col in ["type_materiaal", "genre", "taal", "doelgroep", "interessecluster", "beschikbaar_formaat"]:
    df[col] = df[col].str.strip().str.lower()

# 2) Map spelling variants to one value
df["taal"] = df["taal"].replace({"nl": "nederlands", "ned": "nederlands"})
df["doelgroep"] = df["doelgroep"].str.replace(" ", "").str.upper()       # 'mbo / nt2' -> 'MBO/NT2'
df["beschikbaar_formaat"] = df["beschikbaar_formaat"].replace({"online": "digitaal"})

# 3) Material type: 'boek - roman' and 'boek - thriller' are both books (the subgenre is already in the themes)
df["type_materiaal"] = df["type_materiaal"].replace({
    "boek - roman": "boek",
    "boek - thriller": "boek",
    "krant (papier)": "krant",
    "digitale krant": "krant",
})

# Show the variants after cleaning
for col in cat_cols:
    print(col, "->", df[col].dropna().unique().tolist())

type_materiaal -> ['boek', 'tijdschrift', 'krant', 'online artikel', 'dichtbundel', 'blogpost']
niveau -> ['3F', '2F', '3F+', '2F-3F']
genre -> ['fictie', 'informatief', 'poëzie']
taal -> ['nederlands']
doelgroep -> ['MBO', 'MBO/NT2', 'NT2']
interessecluster -> ['verhalen', 'vrije tijd', 'maatschappij']
uitgever -> ['Educatief Media', 'Leeswereld', 'Uitgeverij Noord']
beschikbaar_formaat -> ['pdf', 'digitaal', 'papier']


### Merge `doelgroep` and `doelgroep_extra`
Both columns describe the target group, but they often disagree (only ~42% match where both are filled).
We merge them by taking the **union** of both: if either column mentions MBO or NT2, that group is kept.
- `MBO` + `NT2` -> `MBO/NT2`
- `MBO` + empty -> `MBO`

After merging, `doelgroep_extra` is deleted.

In [7]:
# --- Merge doelgroep and doelgroep_extra ---
def merge_doelgroep(a, b):
    groups = set()
    for value in (a, b):
        if pd.notna(value):
            groups.update(value.split("/"))           # 'MBO/NT2' -> {'MBO', 'NT2'}
    if not groups:
        return np.nan
    return "/".join(g for g in ["MBO", "NT2"] if g in groups)

# doelgroep_extra is already clean, but we normalize it the same way to be safe
df["doelgroep_extra"] = df["doelgroep_extra"].str.replace(" ", "").str.upper()

both = df["doelgroep"].notna() & df["doelgroep_extra"].notna()
print("Rows where both match:", (df.loc[both, "doelgroep"] == df.loc[both, "doelgroep_extra"]).sum(), "/", both.sum())
print("Missing doelgroep before:", df["doelgroep"].isna().sum())

df["doelgroep"] = [merge_doelgroep(a, b) for a, b in zip(df["doelgroep"], df["doelgroep_extra"])]
df = df.drop(columns="doelgroep_extra")

print("Missing doelgroep after: ", df["doelgroep"].isna().sum())
df["doelgroep"].value_counts(dropna=False)

Rows where both match: 88 / 208
Missing doelgroep before: 6
Missing doelgroep after:  3


doelgroep
MBO/NT2    212
MBO         65
NaN          3
NT2          3
Name: count, dtype: int64

## 5) Fix data types and impossible values
- `aandeel_lange_woorden_pct` is text because some values contain a `%` sign.
- `publicatiejaar` contains `onbekend` (unknown) and the impossible year `2099`.
- `aantal_paginas` contains negative values and 0.
- `gem_woordlengte` contains impossible averages (23-54 letters per word). We recompute those from the description.

In [8]:
# --- Percentage: remove '%' and convert to a number ---
df["aandeel_lange_woorden_pct"] = (
    df["aandeel_lange_woorden_pct"].astype(str).str.replace("%", "", regex=False).astype(float)
)

# --- Publication year: 'onbekend' and future years become NaN ---
df["publicatiejaar"] = pd.to_numeric(df["publicatiejaar"], errors="coerce")   # 'onbekend' -> NaN
df.loc[df["publicatiejaar"] > 2026, "publicatiejaar"] = np.nan                # 2099 -> NaN
df["publicatiejaar"] = df["publicatiejaar"].astype("Int64")                   # whole numbers, NaN allowed

# --- Number of pages: 0 or negative is impossible ---
print("Pages <= 0:", (df["aantal_paginas"] <= 0).sum())
df.loc[df["aantal_paginas"] <= 0, "aantal_paginas"] = np.nan

df[["aandeel_lange_woorden_pct", "publicatiejaar", "aantal_paginas"]].describe()

Pages <= 0: 11


,aandeel_lange_woorden_pct,publicatiejaar,aantal_paginas
count,283.000000,210.0,255.000000
mean,42.430035,2015.533333,184.996078
std,21.840599,8.880996,112.073215
min,0.000000,1998.0,1.000000
25%,28.600000,2013.0,118.500000
50%,35.000000,2018.0,185.000000
75%,50.000000,2023.0,265.500000
max,100.000000,2025.0,400.000000


In [9]:
# --- Average word length: recompute the outliers from the description ---
def avg_word_length(text):
    words = re.findall(r"\w+", str(text))
    return round(np.mean([len(w) for w in words]), 2) if words else np.nan

outliers = df["gem_woordlengte"] > 15     # a Dutch average word length is ~5-8 letters
print("Outliers:", outliers.sum())
print(df.loc[outliers, ["titel", "gem_woordlengte"]])

df.loc[outliers, "gem_woordlengte"] = df.loc[outliers, "omschrijving"].apply(avg_word_length)
print(df.loc[outliers, ["titel", "gem_woordlengte"]])

Outliers: 5
                                                 titel  gem_woordlengte
0                           Het dagboek van Anne Frank            23.48
37                                 Libello voor mannen            30.84
74                                   Het laatste woord            26.52
111                                Jimi Fender Johnson            25.24
148  UWV: aantal banen groeit nauwelijks meer tot 2027            54.00
                                                 titel  gem_woordlengte
0                           Het dagboek van Anne Frank             5.87
37                                 Libello voor mannen             7.71
74                                   Het laatste woord             6.63
111                                Jimi Fender Johnson             6.31
148  UWV: aantal banen groeit nauwelijks meer tot 2027            13.50


## 6) Repair the URL columns
For online articles and blogposts, the URL ended up in `themas` (and junk like `[nos.nl]` in `trefwoorden`).
`bron_url` already contains the same URL, so we can clear these values. We also remove the `[site.nl]` suffix from `bron_url`.

In [10]:
# --- URLs in the wrong column ---
url_in_themas = df["themas"].str.startswith("http", na=False)
print("URLs in themas:", url_in_themas.sum())
print("Of those, bron_url is also filled:", df.loc[url_in_themas, "bron_url"].notna().sum())

df.loc[url_in_themas, "themas"] = np.nan

# trefwoorden that only contain '[site.nl]' are junk
junk_trefwoorden = df["trefwoorden"].str.startswith("[", na=False)
print("Junk trefwoorden:", junk_trefwoorden.sum())
df.loc[junk_trefwoorden, "trefwoorden"] = np.nan

# --- Keep only the actual URL in bron_url ('https://... [nos.nl]' -> 'https://...') ---
df["bron_url"] = df["bron_url"].str.split().str[0]
df.loc[df["bron_url"].notna(), "bron_url"].head()

URLs in themas: 49
Of those, bron_url is also filled: 49
Junk trefwoorden: 2


138    https://www.quest.nl/tech/technologie/a7159937...
139    https://www.autoriteitpersoonsgegevens.nl/actu...
140    https://www.autoriteitpersoonsgegevens.nl/them...
141    https://www.autoriteitpersoonsgegevens.nl/them...
142    https://www.autoriteitpersoonsgegevens.nl/actu...
Name: bron_url, dtype: object

## 7) Clean themes and keywords
`themas` uses `;` as separator and `trefwoorden` uses `; `. We use `;` everywhere, in lowercase.
If `themas` is empty but `trefwoorden` is filled, we copy the keywords.

In [11]:
# --- Themes and keywords with one separator ---
def clean_tags(value):
    if pd.isna(value):
        return np.nan
    tags = [t.strip().lower() for t in str(value).split(";") if t.strip()]
    return ";".join(tags) if tags else np.nan

df["themas"] = df["themas"].apply(clean_tags)
df["trefwoorden"] = df["trefwoorden"].apply(clean_tags)
df["onderwerp"] = df["onderwerp"].str.strip().str.lower()

# Fill empty themes with the keywords
df["themas"] = df["themas"].fillna(df["trefwoorden"])

df[["titel", "themas", "trefwoorden", "onderwerp"]].head(10)

,titel,themas,trefwoorden,onderwerp
0,Het dagboek van Anne Frank,woii;onderduik,woii;onderduik,woii
1,Blauw Water,spanning;stalking,spanning;stalking,spanning
2,Boer vindt vrouw,liefde;relaties,liefde;relaties,liefde
3,Cadeau voor u!,ontmoeting;verbondenheid,ontmoeting;verbondenheid,ontmoeting
4,De biecht,geheimen;schuld,geheimen;schuld,geheimen
5,De familie Blues,familie;humor,familie;humor,familie
6,De favoriete plekken van...,reizen;herinneringen,reizen;herinneringen,reizen
7,De man van de bloemen,ontmoeting;hoop,ontmoeting;hoop,ontmoeting
8,de ontmoeting,ontmoeting;inzicht,ontmoeting;inzicht,ontmoeting
9,De val,oorlog;verlies,oorlog;verlies,oorlog


## 8) Handle duplicates
- There are no fully duplicated rows.
- Some titles (online articles/blogposts) appear multiple times with slightly different values -> we keep the first one.
- 4 `materiaal_id`s are used for two *different* books -> the second one gets a new, unique ID.

In [12]:
# --- Fully duplicated rows ---
print("Fully duplicated rows:", df.duplicated().sum())

# --- Duplicate titles ---
dup_titles = df[df["titel"].duplicated(keep=False)].sort_values("titel")
print(dup_titles[["materiaal_id", "titel", "auteur", "type_materiaal"]])

df = df.drop_duplicates(subset="titel", keep="first").reset_index(drop=True)
print("Shape after removing duplicate titles:", df.shape)

Fully duplicated rows: 0
    materiaal_id                                              titel  \
147      LM-0148          Gaat AI een rol spelen in de rechtspraak?   
177      LM-0178          Gaat AI een rol spelen in de rechtspraak?   
152      LM-0153         Nieuwe stap bij vernieuwing van curriculum   
182      LM-0183         Nieuwe stap bij vernieuwing van curriculum   
140      LM-0141  Transparantie-eisen voor AI: wat betekent dit ...   
176      LM-0177  Transparantie-eisen voor AI: wat betekent dit ...   
139      LM-0140  Vanaf 2 augustus wordt duidelijker of het AI i...   
174      LM-0175  Vanaf 2 augustus wordt duidelijker of het AI i...   
143      LM-0144                            Wat is een rechtsstaat?   
175      LM-0176                            Wat is een rechtsstaat?   
181      LM-0182                            Wat is een rechtsstaat?   

                                     auteur  type_materiaal  
147                           KIJK-redactie  online artikel 

In [13]:
# --- Duplicate IDs: give the second occurrence a new ID ---
dup_ids = df["materiaal_id"].duplicated(keep="first")
print(df.loc[df["materiaal_id"].duplicated(keep=False), ["materiaal_id", "titel", "auteur"]])

next_nr = df["materiaal_id"].str.replace("LM-", "").astype(int).max() + 1
for i in df.index[dup_ids]:
    df.loc[i, "materiaal_id"] = f"LM-{next_nr:04d}"
    next_nr += 1

print("Duplicate IDs left:", df["materiaal_id"].duplicated().sum())

    materiaal_id                                              titel  \
24       LM-0025                       Komt een vrouw bij de dokter   
25       LM-0025                                Mevrouw mijn moeder   
72       LM-0073                De strafzaak tegen Willem Holleeder   
73       LM-0073  Wie zit in de gevangenis? En andere bijzondere...   
120      LM-0121                                     Niet mijn naam   
121      LM-0121                                 Mijn naam is Edino   
216      LM-0223                                  Reis in Rotterdam   
217      LM-0223                              Zolang het foto duurt   

                                      auteur  
24                                     Kluun  
25                              Yvonne Keuls  
72                             Frank Wieland  
73   Charlotte Colman en Stefanie Van Hertem  
120                              Megan Lally  
121                        Edino van Dorsten  
216                            

## 9) Handle missing values
- **Genre**: can be derived from the material type (dichtbundel = poëzie, article/blog = informatief, book = fictie).
- **Categorical columns** (author, level, type, target group, publisher, format): fill with `onbekend`.
- **Numerical columns** (pages, rating, year, ...): we leave these as NaN. Filling them with an average would invent data;
  this can still be done later in the model pipeline if needed.

In [14]:
# --- Missing values ---
print(df.isna().sum()[df.isna().sum() > 0])

auteur                      7
type_materiaal              1
niveau                      6
themas                    177
genre                      26
onderwerp                 175
trefwoorden               177
aantal_paginas             27
geschatte_leestijd_min      9
bron_url                  234
doelgroep                   3
gem_zinslengte_woorden     18
leescomplexiteit_score      6
uitgever                  104
publicatiejaar             73
beschikbaar_formaat        42
gem_beoordeling           100
dtype: int64


In [15]:
# 1) Derive genre from material type
genre_from_type = {
    "boek": "fictie",
    "dichtbundel": "poëzie",
    "online artikel": "informatief",
    "blogpost": "informatief",
}
df["genre"] = df["genre"].fillna(df["type_materiaal"].map(genre_from_type))
print(df["genre"].value_counts(dropna=False))

# 2) Categorical columns -> 'onbekend'
for col in ["auteur", "niveau", "type_materiaal", "genre", "doelgroep", "uitgever", "beschikbaar_formaat"]:
    df[col] = df[col].fillna("onbekend")

print(df.isna().sum()[df.isna().sum() > 0])

genre
fictie         210
informatief     43
poëzie          23
NaN              1
Name: count, dtype: int64
themas                    177
onderwerp                 175
trefwoorden               177
aantal_paginas             27
geschatte_leestijd_min      9
bron_url                  234
gem_zinslengte_woorden     18
leescomplexiteit_score      6
publicatiejaar             73
gem_beoordeling           100
dtype: int64


## 10) Final check and save

In [16]:
# --- Final check ---
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 277 entries, 0 to 276
Data columns (total 26 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   titel                       277 non-null    str    
 1   auteur                      277 non-null    str    
 2   omschrijving                277 non-null    str    
 3   type_materiaal              277 non-null    str    
 4   niveau                      277 non-null    str    
 5   themas                      100 non-null    str    
 6   materiaal_id                277 non-null    str    
 7   genre                       277 non-null    str    
 8   onderwerp                   102 non-null    str    
 9   trefwoorden                 100 non-null    str    
 10  taal                        277 non-null    str    
 11  aantal_paginas              250 non-null    float64
 12  geschatte_leestijd_min      268 non-null    float64
 13  bron_url                    43 non-null     ob

In [17]:
# --- Save the cleaned dataset ---
df.to_csv("../datasets/leesmateriaal_clean.csv", index=False)
print("Saved:", df.shape)

Saved: (277, 26)
